# Lesson 14 Lab — Reductions, Atomics, and Warp Primitives

**Puzzle:** When many values must become one result, which intermediate states should remain thread-local, warp-local, block-local, or global?

This notebook retains one complete RTX 5090 execution.


## Why this matters

A reduction combines values through an associative operation. Efficient GPU reductions usually form a hierarchy: thread-local partials, warp exchange or shared memory, block results, and a final combination. Atomics can combine block results safely, but atomically updating one global accumulator for every input creates maximum contention. Warp shuffle primitives exchange register values without shared memory, provided the active mask is correct.


## 0. Predict before running

1. Predict the fastest route for one scalar sum.
2. Predict how one-bin contention compares with many-bin scatter.
3. Explain where a block barrier is required in a shared-memory tree.

For each prediction, write the observation that would disprove it.


## 1. Theory and mechanism

The notebook compares native `torch.sum` with `scatter_add_` routes that direct the same values into one bin or many bins. It verifies checksums and reports latency. This is not a custom shuffle implementation; it demonstrates why the number and concentration of global updates matter. The theory section then maps that observation onto a hierarchical reduction design.

- Synchronization scope should match the state being shared.
- Hierarchical partial reduction lowers the number of global updates.
- Warp primitives require a correct mask for participating lanes.


## 2. Trace the mechanism

### Mechanism map

```mermaid
flowchart LR
  A["thread-local partials"] --> B["warp reduction"]
  B --> C["block shared state"]
  C --> D["one partial per block"]
  D --> E["final reduction / bounded atomics"]
```


## 3. Inspect the visual boundary

This lesson is driven by a Mermaid mechanism map and executable measurements.


## 4. Inspect the execution environment

The next cell asserts CUDA, records GPU/PyTorch/CUDA identity, fixes the seed, and defines the common event-timing helpers.


In [1]:
LESSON_NO = 14
LESSON_TITLE = 'Reductions, Atomics, and Warp Primitives'

from pathlib import Path
from collections import Counter, deque
import json, math, platform, statistics, sys, time

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chapter 04 retained runs require a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260813 + LESSON_NO
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    pos = (len(ordered) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def cuda_samples(fn, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        stop.record()
        stop.synchronize()
        samples.append(float(start.elapsed_time(stop)))
    return samples

def summary(samples):
    return {
        "median_ms": statistics.median(samples),
        "p95_ms": percentile(samples, 0.95),
        "samples_ms": samples,
    }


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "seed": 20260827
}


## 5. Freeze the experiment

| Role | Frozen value |
|---|---|
| Baseline | optimized `torch.sum` reduction |
| Candidate | one-bin and many-bin `scatter_add_` |
| Held constant | values, dtype, element count, warm-up, and timing |
| Measurements | median latency, checksums, and slowdown versus library reduction |
| Evidence | `pytorch-gpu` |

**Experiment:** Compare library reduction with concentrated and distributed atomic-style updates.


## 6. Inspect the code

Each candidate consumes the same source tensor. The scatter routes allocate destination buffers before timing and clear them per repeat. Output sums are checked within a floating-point tolerance.

Do not run until the code matches the frozen table.


In [2]:
n = 2**24
values = torch.randn(n, device=DEVICE, dtype=torch.float32)
one_idx = torch.zeros(n, device=DEVICE, dtype=torch.int64)
many_bins = 2**16
many_idx = torch.arange(n, device=DEVICE, dtype=torch.int64) % many_bins
one_out = torch.zeros(1, device=DEVICE, dtype=torch.float32)
many_out = torch.zeros(many_bins, device=DEVICE, dtype=torch.float32)

def one_bin():
    one_out.zero_(); one_out.scatter_add_(0, one_idx, values)

def many_bin():
    many_out.zero_(); many_out.scatter_add_(0, many_idx, values)

sum_samples = cuda_samples(lambda: values.sum(), repeats=20)
one_samples = cuda_samples(one_bin, repeats=20)
many_samples = cuda_samples(many_bin, repeats=20)
reference = float(values.sum().item())
one_bin(); many_bin()
one_value = float(one_out.item()); many_value = float(many_out.sum().item())
sum_median = statistics.median(sum_samples)
one_median = statistics.median(one_samples)
many_median = statistics.median(many_samples)
metrics = {
    "sum_median_ms": sum_median, "one_bin_median_ms": one_median,
    "many_bin_median_ms": many_median, "one_bin_slowdown": one_median / sum_median,
    "many_bin_slowdown": many_median / sum_median,
    "checksum_error": max(abs(reference - one_value), abs(reference - many_value)),
    "reference_sum": reference, "one_bin_sum": one_value, "many_bin_sum": many_value,
    "sum_samples_ms": sum_samples, "one_bin_samples_ms": one_samples,
    "many_bin_samples_ms": many_samples,
}
analysis = (
    f"Library sum, one-bin scatter, and many-bin scatter medians were {sum_median:.3f}, "
    f"{one_median:.3f}, and {many_median:.3f} ms. The routes are a hierarchy/contention "
    "probe and may accumulate in different floating-point orders."
)
print(json.dumps(metrics, indent=2))


{
  "sum_median_ms": 0.02195199951529503,
  "one_bin_median_ms": 20.444368362426758,
  "many_bin_median_ms": 0.13411200046539307,
  "one_bin_slowdown": 931.321465645176,
  "many_bin_slowdown": 6.109329602159963,
  "checksum_error": 0.233642578125,
  "reference_sum": -3585.35693359375,
  "one_bin_sum": -3585.590576171875,
  "many_bin_sum": -3585.357421875,
  "sum_samples_ms": [
    0.026464000344276428,
    0.023679999634623528,
    0.022816000506281853,
    0.02252800017595291,
    0.022336000576615334,
    0.021568000316619873,
    0.02380800060927868,
    0.02195199951529503,
    0.021536000072956085,
    0.021888000890612602,
    0.021824000403285027,
    0.021856000646948814,
    0.022112000733613968,
    0.021695999428629875,
    0.022016000002622604,
    0.02195199951529503,
    0.021088000386953354,
    0.021695999428629875,
    0.02195199951529503,
    0.021088000386953354
  ],
  "one_bin_samples_ms": [
    20.451936721801758,
    20.450687408447266,
    20.449535369873047,
   

## 7. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Library sum median | 0.022 ms |
| One-bin median | 20.444 ms |
| Many-bin median | 0.134 ms |
| One-bin slowdown | 931.321x |
| Checksum error | 0.2336 |


## 8. Explain rather than overclaim

Library sum, one-bin scatter, and many-bin scatter medians were 0.022, 20.444, and 0.134 ms. The routes are a hierarchy/contention probe and may accumulate in different floating-point orders.

**Evidence boundary:** CUDA work executed through PyTorch. It does not identify an internal instruction, cache event, or proprietary hardware block without additional profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, metrics, analysis, evidence label, and bounded conclusion, then prints the exact JSON.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 14, "title": 'Reductions, Atomics, and Warp Primitives', "environment": ENV,
    "evidence_label": 'pytorch-gpu', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Start from a trusted library reduction; write a custom hierarchical kernel only when shape, fusion, or output structure justifies it and correctness has an oracle.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 14,
  "title": "Reductions, Atomics, and Warp Primitives",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "seed": 20260827
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "sum_median_ms": 0.02195199951529503,
    "one_bin_median_ms": 20.444368362426758,
    "many_bin_median_ms": 0.13411200046539307,
    "one_bin_slowdown": 931.321465645176,
    "many_bin_slowdown": 6.109329602159963,
    "checksum_error": 0.233642578125,
    "reference_sum": -3585.35693359375,
    "one_bin_sum": -3585.590576171875,
    "many_bin_sum": -3585.357421875,
    "sum_samples_ms": [
      0.026464000344276428,
      0.023679999634623528,
      0.022816000506281853,
      0.02252800017595291,
      0.022336000576615334,
      0.021568000316619873,
      0.02380800060927868,
      0.02195199951529503,
      0.021536000072956085,
      0.021888000890612602,
    

## 10. Make the decision

> Start from a trusted library reduction; write a custom hierarchical kernel only when shape, fusion, or output structure justifies it and correctness has an oracle.

**Failure analysis:** Different kernels may accumulate in different orders, so bitwise output equality is not expected. Scatter is a mechanism probe, not a fair replacement for every reduction.


## 11. Extend the evidence

Implement warp-shuffle and shared-memory block reductions, sweep block sizes, and inspect synchronization plus atomic counters.

See [`README.md`](README.md) for the full explanation and references.
